In [21]:
# Imports

import numpy as np
import pandas as pd
from shared.utils import load_data
from datasets import preprocess_dataset, datasets_types
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from scipy.stats import beta as beta_dist
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score, recall_score, roc_auc_score

seed = 42

from MAB.apollon_mab import ApollonMAB

In [22]:
load_dataset = False
name = "CIC-IDS_2019_MAB"
if not load_dataset:
    # Preprocesar el dataset
    """      """
    df = load_data(
        [
            "./shared/data/CIC_2019/DrDoS_MSSQL.csv"
        ],
        seed
    )
    print("Dataset cargado")
    df_preprocessed = preprocess_dataset(
        df, save=True, dataset_type="CIC_2019", seed=seed, load=load_dataset, name_save=name, name_load=name)
    print("Dataset Preprocesado")
else:
    df_preprocessed = preprocess_dataset(
        pd.DataFrame(), save=True, dataset_type="CIC_2019", seed=seed, load=load_dataset, name_save=name, name_load=name)
    print("Dataset Preprocesado")


c:\Users\AntonioPayá\Desktop\Repos-Doctorado\apollon\shared\utils\load.py:24: DtypeWarning: Columns (85) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path[i])


Dataset cargado
Loading new data
labels: {'DrDoS_MSSQL'}
Dataset Preprocesado


In [ ]:
# Apollon is implemented in MAB/apollon_mab.py. The class previously defined here trained
# every arm on the whole training set, scored arm k only on samples with label k and never
# updated its Beta posteriors (see the module docstring).

In [ ]:
# Train Apollon: pool of RF, DT, NB, LR and MLP, 2 clusters, Thompson sampling per cluster
mab = ApollonMAB(n_clusters=2, random_state=seed)
mab.fit(df_preprocessed.x_train, df_preprocessed.y_train)
print("Clusters/arms that fell back to the global model:\n", mab.fallback_)

In [ ]:
# Test Apollon (a single prediction feeds every metric)
y_pred, selected_arms = mab.predict(df_preprocessed.x_test, return_arms=True)
y_test = np.array([int(y) for y in df_preprocessed.y_test])

print("Arm selection frequencies (RF, DT, NB, LR, MLP):",
      np.bincount(selected_arms, minlength=mab.n_arms) / len(selected_arms))
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
# Attack class (label 1)
print("Detection rate (attack recall):", recall_score(y_test, y_pred))
print("F1 (attack class):", f1_score(y_test, y_pred))
# Macro averages, as reported in results/MAB.txt
print("Recall (macro):", recall_score(y_test, y_pred, average='macro'))
print("F1 (macro):", f1_score(y_test, y_pred, average='macro'))
print("ROC AUC Score:", roc_auc_score(y_test, y_pred))